# **Case Study: Gapminder (Simple)**
Dataset gapminder_simple.csv là phiên bản rút gọn (1000 samples) từ Gapminder. Các cột chính:
*   country: tên quốc gia
*   continent: châu lục
*   year: năm quan sát
*   lifeExp: tuổi thọ trung bình
*   pop: dân số
*   gdpPercap: GDP bình quân đầu người

Mục tiêu: luyện các thao tác retrieval cơ bản (projection, selection, top-N, aggregation,
computed fields) bằng cả Pandas và SQL. Các bạn có thể tải data theo link: https://drive.google.com/file/d/1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs/view

In [1]:
!gdown --id 1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1t68AOeSxixPbDNXHufbYmLfxd0QtoRSs
To: /content/gapminder_simple.csv
100% 50.2k/50.2k [00:00<00:00, 74.4MB/s]


## **1. Import data**

In [2]:
import pandas as pd

df = pd.read_csv('gapminder_simple.csv')
df.head()

,country,year,pop,continent,lifeExp,gdpPercap
0,Myanmar,1962,23634436.0,Asia,45.108,388.000000
1,Ireland,1957,2878220.0,Europe,68.900,5599.077872
2,Jamaica,1977,2156814.0,Americas,70.110,6650.195573
3,Cote d'Ivoire,1987,10761098.0,Africa,54.655,2156.956069
4,Morocco,1997,28529501.0,Africa,67.660,2982.101858


## **2. Projection**

In [6]:
use_cols = ["country", "continent", "year", "lifeExp", "gdpPercap"]
df1 = df[use_cols].copy()

df1.head()

# SQL:
# SELECT country , continent , ‘year ‘, lifeExp , gdpPercap
# FROM gapminder_simple
# LIMIT 10;

,country,continent,year,lifeExp,gdpPercap
0,Myanmar,Asia,1962,45.108,388.000000
1,Ireland,Europe,1957,68.900,5599.077872
2,Jamaica,Americas,1977,70.110,6650.195573
3,Cote d'Ivoire,Africa,1987,54.655,2156.956069
4,Morocco,Africa,1997,67.660,2982.101858


## **Selection**

In [8]:
# Ví dụ: Chỉ lấy Asia, giai đoạn 1990-2007
mask = (df1['continent'] == 'Asia') & (df1['year'] >= 1990) & (df1['year'] <= 2007)
asia_1990_2007 = df.loc[mask,
                        ["country", "continent", "year", "lifeExp", "gdpPercap", "pop"]].copy()

asia_1990_2007.head(10)

# SQL:
# SELECT country , continent , ‘year ‘, lifeExp , gdpPercap , pop
# FROM gapminder_simple
# WHERE continent = ’Asia ’
#   AND ‘year ‘ BETWEEN 1990 AND 2007
# LIMIT 10;

,country,continent,year,lifeExp,gdpPercap,pop
7,Taiwan,Asia,1997,75.250,20206.820980,2.162860e+07
19,Nepal,Asia,2002,61.340,1057.206311,2.587392e+07
28,Mongolia,Asia,1997,63.625,1902.252100,2.494803e+06
40,India,Asia,2007,64.698,2452.210407,1.110396e+09
73,India,Asia,2002,62.879,1746.769454,1.034173e+09
80,Israel,Asia,1992,76.930,18051.522540,4.936550e+06
224,China,Asia,2002,72.028,3119.280896,1.280400e+09
234,Vietnam,Asia,1997,70.672,1385.896769,7.604900e+07
240,Singapore,Asia,2002,78.770,36023.105400,4.197776e+06
247,Myanmar,Asia,1992,59.320,347.000000,4.054654e+07


## **Ordering & Top-N**

In [11]:
# Lấy 10 quốc gia có gpdPercap cao nhất trong năm 2007
df_2007 = df[df["year"] == 2007].copy()

top10_gdpPercap_2007 = (
    df_2007.sort_values("gdpPercap", ascending=False)
    [["country", "continent", "year", "gdpPercap", "lifeExp"]]
    .head(10)
)

top10_gdpPercap_2007

# SQL:
# SELECT country , continent , ‘year ‘, gdpPercap , lifeExp
# FROM gapminder_simple
# WHERE ‘year ‘ = 2007
# ORDER BY gdpPercap DESC
# LIMIT 10;

,country,continent,year,gdpPercap,lifeExp
483,Ireland,Europe,2007,40675.99635,78.885
354,Switzerland,Europe,2007,37506.41907,81.701
988,Netherlands,Europe,2007,36797.93332,79.762
217,Canada,Americas,2007,36319.23501,80.653
900,Iceland,Europe,2007,36180.78919,81.757
554,Austria,Europe,2007,36126.49270,79.829
643,Denmark,Europe,2007,35278.41874,78.332
986,Australia,Oceania,2007,34435.36744,81.235
133,Finland,Europe,2007,33207.08440,79.313
664,United Kingdom,Europe,2007,33203.26128,79.425


## **3. Aggregation (Simple)**

Mục tiêu: tổng hợp theo châu lục trong năm 2007 với 2 chỉ số cơ bản: tuổi thọ trung bình và GDP/người trung bình

In [15]:
tmp = df[df["year"] == 2007].copy()

agg_continent_2007 = (
    tmp.groupby("continent", as_index=False)
    .agg(
        avg_lifeExp=("lifeExp", "mean"),
        avg_gdpPercap=("gdpPercap", "mean"),
    )
    .sort_values("avg_gdpPercap", ascending=False)
)

agg_continent_2007

# SQL:
# SELECT
#   continent ,
#   AVG ( lifeExp ) AS avg_lifeExp ,
#   AVG ( gdpPercap ) AS avg_gdpPercap
# FROM gapminder_simple
# WHERE ‘year ‘ = 2007
# GROUP BY continent
# ORDER BY avg_gdpPercap DESC ;

,continent,avg_lifeExp,avg_gdpPercap
4,Oceania,81.235000,34435.367440
3,Europe,77.739950,24975.915126
1,Americas,75.119923,12113.551428
2,Asia,69.328118,8127.222843
0,Africa,54.895667,3660.861747


## **Computed fields**

Mục tiêu: tạo GDP xấp xỉ theo công thức GDP = pop * gdpPercap, rồi lấy top theo GDP trong năm 2007.

In [17]:
tmp = df[df["year"] == 2007].copy()
tmp["gdp"] = tmp["pop"] * tmp["gdpPercap"]

top10_gdp_2007 = (
    tmp.sort_values("gdp", ascending=False)
    [["country", "continent", "year", "pop", "gdpPercap", "gdp", "lifeExp"]]
    .head(10)
)

top10_gdp_2007

# SQL
# SELECT
#   country , continent , ‘year ‘,
#   pop , gdpPercap ,
#   (pop * gdpPercap ) AS gdp ,
#   lifeExp
# FROM gapminder_simple
# WHERE ‘year ‘ = 2007
# ORDER BY gdp DESC
# LIMIT 10;

,country,continent,year,pop,gdpPercap,gdp,lifeExp
736,China,Asia,2007,1.318683e+09,4959.114854,6.539501e+12,72.961
854,Japan,Asia,2007,1.274680e+08,31656.068060,4.035135e+12,82.603
40,India,Asia,2007,1.110396e+09,2452.210407,2.722925e+12,64.698
344,Germany,Europe,2007,8.240100e+07,32170.374420,2.650871e+12,79.406
664,United Kingdom,Europe,2007,6.077624e+07,33203.261280,2.017969e+12,79.425
563,Brazil,Americas,2007,1.900106e+08,9065.800825,1.722599e+12,72.390
217,Canada,Americas,2007,3.339014e+07,36319.235010,1.212704e+12,80.653
967,Spain,Europe,2007,4.044819e+07,28821.063700,1.165760e+12,80.941
986,Australia,Oceania,2007,2.043418e+07,34435.367440,7.036584e+11,81.235
988,Netherlands,Europe,2007,1.657061e+07,36797.933320,6.097643e+11,79.762
